In [30]:
# 🔹 Install these packages if you haven't already:
# pip install langchain langchain-community sentence-transformers faiss-cpu transformers gradio

from langchain_community.document_loaders import DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFacePipeline
from langchain.chains import RetrievalQA

from transformers import pipeline
import gradio as gr

# Step 1: Load markdown files
loader = DirectoryLoader("/content/drive/MyDrive/know-base", glob="**/*.md")
documents = loader.load()

# Step 2: Chunk the documents
splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=50)
chunks = splitter.split_documents(documents)

# (Optional) Print a few chunks
print(f"🔹 Total chunks: {len(chunks)}")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i+1} ---\n{chunk.page_content[:300]}...")

# Step 3: Embeddings using open-source model
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding_model)

# Step 4: Load open-source LLM using Hugging Face Transformers
llm_pipeline = pipeline(
    task="text2text-generation",
    model="google/flan-t5-base",   # small & CPU-friendly
    max_new_tokens=256
)

llm = HuggingFacePipeline(pipeline=llm_pipeline)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    return_source_documents=False  # Optional, change to True if you want to show sources
)

# Step 6: Gradio Interface
def ask_hammad_ai(question):
    try:
        formatted_question = f"Question: {question}\nAnswer:"
        result = qa_chain.run(formatted_question)
        return result
    except Exception as e:
        return f"❌ Error: {str(e)}"

gr.Interface(
    fn=ask_hammad_ai,
    inputs="text",
    outputs="text",
    title="🤖 Hammad's Personal AI Assistant",
    description="Ask anything about Hammad Farooq — his courses, projects, and work experience."
).launch(inline=True)


🔹 Total chunks: 37

--- Chunk 1 ---
About This Knowledge Base

This knowledge base is a structured collection of professional, academic, and project-related documents about Hammad Farooq.

📌 Who is Hammad Farooq?

Name: Hammad Farooq

Current Education: BS in Data Science...

--- Chunk 2 ---
Current Education: BS in Data Science

Semester: 6th Semester

University: Punjab University (PU)

📂 Folder Structure

/courses/...

--- Chunk 3 ---
📂 Folder Structure

/courses/

Contains summaries of major courses Hammad has completed, primarily through Udemy and other self-paced platforms. These include: - Python Bootcamp - Flask with Python - Data Science Bootcamp - LLM Engineering Course...


Device set to use cuda:0


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3c99dd8cdc55cd94b0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [20]:
!pip install -U langchain-huggingface
